In [3]:
import pandas as pd
import cloudscraper
from datetime import datetime, timedelta, date
from dateutil.relativedelta import relativedelta
import boto3
from io import BytesIO
import time
import random
import json

BUCKET_NAME = "afdal-idx-stock-data-s3"
METADATA_KEY = "metadata/etl_control.json"

In [47]:
def read_df_from_s3(bucket_name, key):
    s3 = boto3.client("s3")
    response = s3.get_object(
        Bucket=bucket_name,
        Key=key
    )
    buffer = BytesIO(response["Body"].read())
    return pd.read_parquet(buffer)

In [3]:
def get_month_list(start_date, end_date):
    months = set()
    current_date = start_date
    while current_date <= end_date:
        year = current_date.year
        month = current_date.month
        year_month = year * 100 + month
        months.add(year_month)
        current_date += timedelta(days=1)
    return sorted(months)

In [5]:
def load_raw(start_date, end_date):
    months = get_month_list(start_date, end_date)
    s3 = boto3.client("s3")

    dfs = {}
    for i, month in enumerate(months):
        mm = f'{month%100:02d}'
        yyyy = f'{int(month/100)}'
        # print(f'{mm}-{yyyy},{i}')
        if i!=len(months)-1:
            key = f'raw/idx/year={yyyy}/month={mm}/data.parquet'
            dfs[key] = read_df_from_s3(BUCKET_NAME, key)
        else:
            first_date = f'{yyyy}-{mm}-01' if len(months)>1 else f'{start_date}'
            list_date = pd.date_range(first_date, f'{end_date}')
            for j in list_date:
                date = j.date()
                key = f'raw/idx/year={yyyy}/month={mm}/daily/date={date}/data.parquet'
                dfs[key] = read_df_from_s3(BUCKET_NAME, key)

    return dfs

In [58]:
star_date = datetime.strptime('2026-08-30', '%Y-%m-%d').date()
end_date = datetime.strptime('2026-09-01', '%Y-%m-%d').date()
dfs = load_raw(star_date, end_date)
# print(1)

In [7]:
standarize_column_name ={
    "IDStockSummary" : "stock_summary_id",
    "Date" : "trade_date",
    "StockCode" : "stock_code",
    "StockName" : "stock_name",
    "Remarks" : "remarks",
    "DelistingDate" : "delisting_date",
    "Previous" : "previous_price",
    "OpenPrice" : "open_price",
    "FirstTrade" : "first_trade_price",
    "High" : "high_price",
    "Low" : "low_price",
    "Close" : "close_price",
    "Change" : "price_change",
    "Volume" : "volume",
    "Value" : "trading_value",
    "Frequency" : "frequency",
    "IndexIndividual" : "individual_index",
    "Offer" : "offer_price",
    "OfferVolume" : "offer_volume",
    "Bid": "bid_price",
    "BidVolume": "bid_volume",
    "ListedShares": "listed_shares",
    "TradebleShares": "tradeable_shares",
    "WeightForIndex": "index_weight",
    "ForeignSell": "foreign_sell",
    "ForeignBuy": "foreign_buy",
    "NonRegularVolume": "non_regular_volume",
    "NonRegularValue": "non_regular_value",
    "NonRegularFrequency": "non_regular_frequency",
}

In [92]:
df = dfs['raw/idx/year=2026/month=08/data.parquet']
df.head()

,No,stock_summary_id,trade_date,stock_code,stock_name,remarks,previous_price,open_price,first_trade_price,high_price,...,tradeable_shares,index_weight,foreign_sell,foreign_buy,delisting_date,non_regular_volume,non_regular_value,non_regular_frequency,persen,percentage
0,1,4065341,2026-08-03,AADI,Adaro Andalan Indonesia Tbk.,--MO1SD0F10000A121------------,9225.0,9300.0,9300.0,9300.0,...,7786891760,1.498977e+09,2396100,503600,NaT,147964,1.339833e+09,16,None,None
1,2,4065342,2026-08-03,AALI,Astra Agro Lestari Tbk.,--MO113E100000D232------------,6875.0,6900.0,6900.0,7175.0,...,1924688333,3.903268e+08,1111300,738100,NaT,0,0.000000e+00,0,None,None
2,3,4065343,2026-08-03,ABBA,Mahaka Media Tbk.,--U-4100000000E614-E---------X,30.0,0.0,0.0,0.0,...,3935892857,1.298451e+09,0,0,NaT,0,0.000000e+00,0,None,None
3,4,4065344,2026-08-03,ABDA,Asuransi Bina Dana Arta Tbk.,--UO2105000000G412------------,3500.0,0.0,0.0,3530.0,...,620806680,5.947328e+07,0,0,NaT,0,0.000000e+00,0,None,None
4,5,4065345,2026-08-03,ABMM,ABM Investama Tbk.,--MO1135000000A121------------,2510.0,2550.0,2550.0,2580.0,...,2753165000,4.129748e+08,14600,197600,NaT,16200,4.357800e+07,1,None,None


In [9]:
def rename_column(dataframe):
    dataframe.rename(columns=standarize_column_name, inplace=True)
    return dataframe

In [1]:
df = rename_column(dfs['raw/idx/year=2026/month=08/data.parquet'])
df.head(10)

NameError: name 'rename_column' is not defined

In [69]:
integer_columns = [
    "stock_summary_id",
    "volume",
    "frequency",
    "offer_volume",
    "bid_volume",
    "listed_shares",
    "tradeable_shares",
    "foreign_sell",
    "foreign_buy",
    "non_regular_volume",
    "non_regular_frequency",
]

float_columns = [
    "previous_price",
    "open_price",
    "first_trade_price",
    "high_price",
    "low_price",
    "close_price",
    "price_change",
    "trading_value",
    "individual_index",
    "offer_price",
    "bid_price",
    "index_weight",
    "non_regular_value",
]

string_columns = [
    "stock_code",
    "stock_name",
    "remarks",
]

date_columns = [
    "trade_date",
    "delisting_date",
]


def cast_datatype(dataframe):
    # Integer
    for column in integer_columns:
        dataframe[column] = pd.to_numeric(
            dataframe[column],
            errors="coerce"
        ).astype("Int64")

    # Float
    for column in float_columns:
        dataframe[column] = pd.to_numeric(
            dataframe[column],
            errors="coerce"
        ).astype("float64")

    # String
    for column in string_columns:
        dataframe[column] = dataframe[column].astype("string")

    # Date
    for column in date_columns:
        dataframe[column] = pd.to_datetime(
            dataframe[column],
            errors="coerce"
        ).dt.date

    return dataframe

In [85]:
def quality_check(dataframe):
    errors = []

    # 1. Cleaning awal: Ubah nilai negatif/sentinel (< 0) jadi NaN untuk kolom numerik
    non_negative_columns = [
        "previous_price", "open_price", "first_trade_price", "high_price",
        "low_price", "close_price", "volume", "trading_value", "frequency",
        "offer_price", "offer_volume", "bid_price", "bid_volume",
        "listed_shares", "tradeable_shares", "foreign_sell", "foreign_buy",
        "non_regular_volume", "non_regular_value", "non_regular_frequency",
    ]

    for col in non_negative_columns:
        if col in dataframe.columns:
            invalid_count = (dataframe[col] < 0).sum()
            if invalid_count > 0:
                logging.warning(f"Warning: {invalid_count} baris bernilai < 0 pada kolom '{col}' diubah menjadi NaN.")
                dataframe.loc[dataframe[col] < 0, col] = np.nan

    # 2. Required / NOT NULL check (Fatal Error)
    required_columns = ["trade_date", "stock_code"]
    null_columns = dataframe[required_columns].isna().any()
    if null_columns.any():
        columns = null_columns[null_columns].index.tolist()
        errors.append(f"NULL ditemukan pada required column: {columns}")

    # 3. Duplicate check (Fatal Error)
    duplicate_mask = dataframe.duplicated(subset=required_columns, keep=False)
    if duplicate_mask.any():
        errors.append(
            f"Terdapat duplikat berdasarkan ({required_columns[0]}, {required_columns[1]}): {duplicate_mask.sum()} rows"
        )

    # 4. OHLC validation (Setelah dibersihkan jadi NaN, validasi ini otomatis aman dari angka negatif)
    ohlc_validations = {
        "High < Low": dataframe["high_price"] < dataframe["low_price"],
        "Open < Low": dataframe["open_price"] < dataframe["low_price"],
        "Open > High": dataframe["open_price"] > dataframe["high_price"],
        "Close < Low": dataframe["close_price"] < dataframe["low_price"],
        "Close > High": dataframe["close_price"] > dataframe["high_price"],
    }

    for rule, invalid_mask in ohlc_validations.items():
        if invalid_mask.any():
            #Warning anomali
            logging.warning(f"OHLC Warning - {rule}: {invalid_mask.sum()} rows terdeteksi anomali (kemungkinan saham FCA).")

    # 5. Trade date future check (Fatal Error)
    future_date_mask = dataframe["trade_date"] > pd.Timestamp.today().date()
    if future_date_mask.any():
        errors.append(f"trade_date berada di masa depan: {future_date_mask.sum()} rows")

    # Final result
    if errors:
        raise ValueError(
            "Quality check failed (Structural Error):\n- "
            + "\n- ".join(errors)
        )

    print("Quality check passed")
    return dataframe

In [86]:
df = quality_check(df)

Quality check passed


In [ ]:

# %%
import logging
from io import BytesIO
from datetime import datetime, timedelta, date

import boto3
import numpy as np
import pandas as pd

BUCKET_NAME = "afdal-idx-stock-data-s3"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logger = logging.getLogger(__name__)

#Mapping column
STANDARDIZE_COLUMN_MAP = {
    "IDStockSummary": "stock_summary_id",
    "Date": "trade_date",
    "StockCode": "stock_code",
    "StockName": "stock_name",
    "Remarks": "remarks",
    "DelistingDate": "delisting_date",

    "Previous": "previous_price",
    "OpenPrice": "open_price",
    "FirstTrade": "first_trade_price",
    "High": "high_price",
    "Low": "low_price",
    "Close": "close_price",
    "Change": "price_change",

    "Volume": "volume",
    "Value": "trading_value",
    "Frequency": "frequency",

    "IndexIndividual": "individual_index",

    "Offer": "offer_price",
    "OfferVolume": "offer_volume",
    "Bid": "bid_price",
    "BidVolume": "bid_volume",

    "ListedShares": "listed_shares",
    "TradebleShares": "tradeable_shares",
    "WeightForIndex": "index_weight",

    "ForeignSell": "foreign_sell",
    "ForeignBuy": "foreign_buy",

    "NonRegularVolume": "non_regular_volume",
    "NonRegularValue": "non_regular_value",
    "NonRegularFrequency": "non_regular_frequency",
}


EXPECTED_SOURCE_COLUMNS = set(STANDARDIZE_COLUMN_MAP.keys())

EXPECTED_CURATED_COLUMNS = list(STANDARDIZE_COLUMN_MAP.values())


# Data type
INTEGER_COLUMNS = [
    "stock_summary_id",
    "volume",
    "frequency",
    "offer_volume",
    "bid_volume",
    "listed_shares",
    "tradeable_shares",
    "foreign_sell",
    "foreign_buy",
    "non_regular_volume",
    "non_regular_frequency",
]

FLOAT_COLUMNS = [
    "previous_price",
    "open_price",
    "first_trade_price",
    "high_price",
    "low_price",
    "close_price",
    "price_change",
    "trading_value",
    "individual_index",
    "offer_price",
    "bid_price",
    "index_weight",
    "non_regular_value",
]

STRING_COLUMNS = [
    "stock_code",
    "stock_name",
    "remarks",
]

DATE_COLUMNS = [
    "trade_date",
    "delisting_date",
]


# S3
def read_df_from_s3(bucket_name, key):

    s3_client = boto3.client("s3")

    response = s3_client.get_object(
        Bucket=bucket_name,
        Key=key
    )

    buffer = BytesIO(response["Body"].read())

    return pd.read_parquet(buffer)

def get_month_list(start_date, end_date):
    months = set()

    current_date = start_date

    while current_date <= end_date:
        year_month = current_date.year * 100 + current_date.month
        months.add(year_month)

        current_date += timedelta(days=1)

    return sorted(months)


def load_raw(start_date, end_date):
    months = get_month_list(start_date, end_date)

    s3 = boto3.client("s3")

    dfs = {}

    for i, year_month in enumerate(months):

        month = year_month % 100
        year = year_month // 100

        mm = f"{month:02d}"
        yyyy = f"{year:04d}"


        if i != len(months) - 1:
            key = f"raw/idx/year={yyyy}/month={mm}/data.parquet"

            logger.info(f"Loading raw: {key}")

            dfs[key] = read_df_from_s3(BUCKET_NAME, key)

        else:
            if len(months) > 1:
                first_date = date(year, month, 1)
            else:
                first_date = start_date

            date_range = pd.date_range(
                first_date,
                end_date
            )

            for current_date in date_range:
                current_date = current_date.date()
                key = f"raw/idx/year={yyyy}/month={mm}/daily/date={current_date}/data.parquet"

                logger.info(f"Loading raw: {key}")

                dfs[key] = read_df_from_s3(BUCKET_NAME,key)

    logger.info(f"Loaded {len(dfs)} raw files")

    return dfs


# Validation
def validate_source_schema(dataframe):
    actual_columns = set(dataframe.columns)

    missing_columns = EXPECTED_SOURCE_COLUMNS - actual_columns
    unexpected_columns = actual_columns - EXPECTED_SOURCE_COLUMNS

    if missing_columns:
        raise ValueError(
            f"Source schema validation failed. Missing columns: {sorted(missing_columns)}"
        )

    if unexpected_columns:
        logger.warning(
            f"Unexpected source columns detected: {sorted(unexpected_columns)}"
        )

    logger.info("Source schema validation passed")

    return dataframe


# Rename columns
def rename_columns(dataframe):
    return dataframe.rename(
        columns=STANDARDIZE_COLUMN_MAP
    )


# Data type casting
def cast_numeric_column(dataframe, column, dtype):
    original = dataframe[column]

    original_null = original.isna()

    converted = pd.to_numeric(
        original,
        errors="coerce"
    )

    conversion_error_mask = (
        ~original_null &
        converted.isna()
    )

    conversion_error_count = conversion_error_mask.sum()

    if conversion_error_count > 0:

        bad_values = original[
            conversion_error_mask
        ].unique()

        raise ValueError(
            f"Datatype conversion failed for column "
            f"'{column}'. "
            f"{conversion_error_count} invalid values. "
            f"Examples: {bad_values[:5]}"
        )

    dataframe[column] = converted.astype(dtype)

    return dataframe


def cast_datatypes(dataframe):
    """
    Apply the curated data type contract.
    """

    # --------------------------------------------------------
    # Integer
    # --------------------------------------------------------

    for column in INTEGER_COLUMNS:

        dataframe = cast_numeric_column(
            dataframe,
            column,
            "Int64"
        )

    # --------------------------------------------------------
    # Float
    # --------------------------------------------------------

    for column in FLOAT_COLUMNS:

        dataframe = cast_numeric_column(
            dataframe,
            column,
            "float64"
        )

    # --------------------------------------------------------
    # String
    # --------------------------------------------------------

    for column in STRING_COLUMNS:

        dataframe[column] = (
            dataframe[column]
            .astype("string")
            .str.strip()
        )

    # --------------------------------------------------------
    # Date / datetime
    # --------------------------------------------------------

    for column in DATE_COLUMNS:

        dataframe[column] = pd.to_datetime(
            dataframe[column],
            errors="coerce"
        ).dt.date

    logger.info("Datatype casting passed")

    return dataframe


# ============================================================
# DATA QUALITY CHECK
# ============================================================

def quality_check(dataframe):
    errors = []

    # ========================================================
    # 1. Required columns
    # ========================================================

    required_columns = [
        "trade_date",
        "stock_code",
    ]

    for column in required_columns:

        if dataframe[column].isna().any():

            count = dataframe[column].isna().sum()

            errors.append(
                f"NULL found in required column "
                f"'{column}': {count} rows"
            )

    # ========================================================
    # 2. Required string values must not be empty
    # ========================================================

    empty_stock_code = (
        dataframe["stock_code"]
        .fillna("")
        .str.strip()
        .eq("")
    )

    if empty_stock_code.any():

        count = empty_stock_code.sum()

        errors.append(
            f"Empty stock_code: {count} rows"
        )

    # ========================================================
    # 3. Duplicate grain
    # ========================================================

    grain_columns = [
        "trade_date",
        "stock_code",
    ]

    duplicate_mask = dataframe.duplicated(
        subset=grain_columns,
        keep=False
    )

    if duplicate_mask.any():

        duplicate_count = duplicate_mask.sum()

        errors.append(
            f"Duplicate grain detected for "
            f"({', '.join(grain_columns)}): "
            f"{duplicate_count} rows"
        )

    # ========================================================
    # 4. Non-negative values
    #
    # IDX uses negative sentinel values such as -9999999
    # for fields that cannot logically be negative.
    #
    # Treat them as NULL.
    # ========================================================

    non_negative_columns = [
        "previous_price",
        "open_price",
        "first_trade_price",
        "high_price",
        "low_price",
        "close_price",

        "volume",
        "trading_value",
        "frequency",

        "offer_price",
        "offer_volume",
        "bid_price",
        "bid_volume",

        "listed_shares",
        "tradeable_shares",

        "foreign_sell",
        "foreign_buy",

        "non_regular_volume",
        "non_regular_value",
        "non_regular_frequency",
    ]

    for column in non_negative_columns:

        invalid_mask = dataframe[column] < 0

        if invalid_mask.any():

            count = invalid_mask.sum()

            logger.warning(
                f"Negative sentinel values detected in "
                f"'{column}': {count} rows. "
                f"Converting to NULL."
            )

            dataframe.loc[
                invalid_mask,
                column
            ] = pd.NA

    # ========================================================
    # 5. OHLC consistency
    # ========================================================

    ohlc_validations = {

        "High < Low":
            dataframe["high_price"] <
            dataframe["low_price"],

        "Open < Low":
            dataframe["open_price"] <
            dataframe["low_price"],

        "Open > High":
            dataframe["open_price"] >
            dataframe["high_price"],

        "Close < Low":
            dataframe["close_price"] <
            dataframe["low_price"],

        "Close > High":
            dataframe["close_price"] >
            dataframe["high_price"],
    }

    for rule, invalid_mask in ohlc_validations.items():

        count = invalid_mask.sum()

        if count > 0:

            logger.warning(
                f"OHLC anomaly - {rule}: "
                f"{count} rows"
            )

    # ========================================================
    # 6. Future trade date
    # ========================================================

    today = pd.Timestamp.today().date

    future_date_mask = (
        dataframe["trade_date"] > today
    )

    if future_date_mask.any():

        count = future_date_mask.sum()

        errors.append(
            f"trade_date in the future: "
            f"{count} rows"
        )

    # ========================================================
    # 7. Final result
    # ========================================================

    if errors:

        raise ValueError(
            "Quality check failed:\n- "
            + "\n- ".join(errors)
        )

    logger.info("Quality check passed")

    return dataframe

# ============================================================
# FINAL CURATED TRANSFORMATION
# ============================================================

def transform_to_curated(dataframe):
    """
    Complete RAW → CURATED transformation.
    """

    logger.info("Starting RAW → CURATED transformation")

    # 1. Validate source contract
    dataframe = validate_source_schema(dataframe)

    # 2. Standardize column names
    dataframe = rename_columns(dataframe)

    # 3. Select only curated columns
    #
    # This intentionally removes source-only fields such as:
    # No, persen, percentage, or future unexpected fields.
    dataframe = dataframe[EXPECTED_CURATED_COLUMNS].copy()

    # 4. Cast data types
    dataframe = cast_datatypes(dataframe)

    # 5. Quality checks
    dataframe = quality_check(dataframe)

    logger.info(
        f"RAW → CURATED completed. "
        f"Rows: {len(dataframe):,}, "
        f"Columns: {len(dataframe.columns)}"
    )

    return dataframe

dfs = load_raw(
    datetime.strptime("2026-08-30", "%Y-%m-%d").date(),
    datetime.strptime("2026-09-01", "%Y-%m-%d").date()
)

df_raw = dfs[
    "raw/idx/year=2026/month=08/data.parquet"
]

df_curated = transform_to_curated(df_raw)



In [95]:
dfs = load_raw(
    datetime.strptime("2026-08-30", "%Y-%m-%d").date(),
    datetime.strptime("2026-09-01", "%Y-%m-%d").date()
)

df_raw = dfs[
    "raw/idx/year=2026/month=08/data.parquet"
]

df_curated = transform_to_curated(df_raw)

In [23]:
import logging
from io import BytesIO
from datetime import datetime, timedelta, date

import boto3
import pandas as pd
from botocore.exceptions import ClientError

BUCKET_NAME = "afdal-idx-stock-data-s3"

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

# Mapping column
STANDARDIZE_COLUMN_MAP = {
    "IDStockSummary": "stock_summary_id",
    "Date": "trade_date",
    "StockCode": "stock_code",
    "StockName": "stock_name",
    "Remarks": "remarks",
    "DelistingDate": "delisting_date",

    "Previous": "previous_price",
    "OpenPrice": "open_price",
    "FirstTrade": "first_trade_price",
    "High": "high_price",
    "Low": "low_price",
    "Close": "close_price",
    "Change": "price_change",

    "Volume": "volume",
    "Value": "trading_value",
    "Frequency": "frequency",

    "IndexIndividual": "individual_index",

    "Offer": "offer_price",
    "OfferVolume": "offer_volume",
    "Bid": "bid_price",
    "BidVolume": "bid_volume",

    "ListedShares": "listed_shares",
    "TradebleShares": "tradeable_shares",
    "WeightForIndex": "index_weight",

    "ForeignSell": "foreign_sell",
    "ForeignBuy": "foreign_buy",

    "NonRegularVolume": "non_regular_volume",
    "NonRegularValue": "non_regular_value",
    "NonRegularFrequency": "non_regular_frequency",
}

EXPECTED_SOURCE_COLUMNS = set(STANDARDIZE_COLUMN_MAP.keys())
EXPECTED_CURATED_COLUMNS = list(STANDARDIZE_COLUMN_MAP.values())

# Data type
INTEGER_COLUMNS = [
    "stock_summary_id", "volume", "frequency", "offer_volume", "bid_volume",
    "listed_shares", "tradeable_shares", "foreign_sell", "foreign_buy",
    "non_regular_volume", "non_regular_frequency",
]

FLOAT_COLUMNS = [
    "previous_price", "open_price", "first_trade_price", "high_price", "low_price",
    "close_price", "price_change", "trading_value", "individual_index",
    "offer_price", "bid_price", "index_weight", "non_regular_value",
]

STRING_COLUMNS = ["stock_code", "stock_name", "remarks"]
DATE_COLUMNS = ["trade_date", "delisting_date"]


# S3
# def read_df_from_s3(bucket_name, key, s3_client=None):
#     s3_client = s3_client or boto3.client("s3")
#     response = s3_client.get_object(Bucket=bucket_name, Key=key)
#     buffer = BytesIO(response["Body"].read())
#     return pd.read_parquet(buffer)

def read_df_from_s3(bucket_name, key, s3_client=None):
    s3_client = s3_client or boto3.client("s3")
    try:
        response = s3_client.get_object(
            Bucket=bucket_name,
            Key=key,
        )
    except ClientError as e:
        if e.response["Error"]["Code"] in ("NoSuchKey", "404"):
            return None
        raise

    return pd.read_parquet(BytesIO(response["Body"].read()))



def upload_df_to_s3(df, bucket_name, key, s3_client=None):
    s3_client = s3_client or boto3.client("s3")
    buffer = BytesIO()
    df.to_parquet(buffer, index=False, engine="pyarrow")
    s3_client.put_object(Bucket=bucket_name, Key=key, Body=buffer.getvalue())


def save_curated_to_s3(dfs:dict):
    s3_client = boto3.client("s3")
    for key,value in dfs.items():
        curated_key = key.replace("raw/", "curated/", 1)
        logger.info(f"Uploading curated: {curated_key} ({len(value)} rows)")
        upload_df_to_s3(value, BUCKET_NAME, curated_key, s3_client)


def get_month_list(start_date, end_date):
    months = set()
    current_date = start_date
    while current_date <= end_date:
        months.add(current_date.year * 100 + current_date.month)
        current_date += timedelta(days=1)
    return sorted(months)


def load_raw(start_date, end_date):
    """Load raw data for a date range and return it as one concatenated DataFrame."""
    months = get_month_list(start_date, end_date)
    s3_client = boto3.client("s3")
    dfs = {}

    for i, year_month in enumerate(months):
        month, year = year_month % 100, year_month // 100
        mm, yyyy = f"{month:02d}", f"{year:04d}"

        if i != len(months) - 1:
            key = f"raw/idx/year={yyyy}/month={mm}/data.parquet"
            logger.info(f"Loading raw: {key}")
            dfs[key] = read_df_from_s3(BUCKET_NAME, key, s3_client)
        else:
            first_date = date(year, month, 1) if len(months) > 1 else start_date
            for current_date in pd.date_range(first_date, end_date):
                current_date = current_date.date()
                key = f"raw/idx/year={yyyy}/month={mm}/daily/date={current_date}/data.parquet"
                logger.info(f"Loading raw: {key}")

                df = read_df_from_s3(BUCKET_NAME, key, s3_client)
                if df is not None:
                    dfs[key] = df
                else:
                    logger.info(f"No data found: {key}")

    logger.info(f"Loaded {len(dfs)} raw files")
    return dfs


# Validation
def validate_source_schema(dataframe):
    actual_columns = set(dataframe.columns)
    missing_columns = EXPECTED_SOURCE_COLUMNS - actual_columns
    unexpected_columns = actual_columns - EXPECTED_SOURCE_COLUMNS

    if missing_columns:
        raise ValueError(f"Source schema validation failed. Missing columns: {sorted(missing_columns)}")

    if unexpected_columns:
        logger.warning(f"Unexpected source columns detected: {sorted(unexpected_columns)}")

    logger.info("Source schema validation passed")
    return dataframe


def rename_columns(dataframe):
    return dataframe.rename(columns=STANDARDIZE_COLUMN_MAP)


# Data type casting
def cast_numeric_column(dataframe, column, dtype):
    original = dataframe[column]
    original_null = original.isna()
    converted = pd.to_numeric(original, errors="coerce")

    conversion_error_mask = ~original_null & converted.isna()
    conversion_error_count = conversion_error_mask.sum()

    if conversion_error_count > 0:
        bad_values = original[conversion_error_mask].unique()
        raise ValueError(
            f"Datatype conversion failed for column '{column}'. "
            f"{conversion_error_count} invalid values. Examples: {bad_values[:5]}"
        )

    dataframe[column] = converted.astype(dtype)
    return dataframe


def cast_datatypes(dataframe):
    """Apply the curated data type contract."""
    for column in INTEGER_COLUMNS:
        dataframe = cast_numeric_column(dataframe, column, "Int64")

    for column in FLOAT_COLUMNS:
        dataframe = cast_numeric_column(dataframe, column, "float64")

    for column in STRING_COLUMNS:
        dataframe[column] = dataframe[column].astype("string").str.strip()

    for column in DATE_COLUMNS:
        dataframe[column] = pd.to_datetime(dataframe[column], errors="coerce").dt.date

    logger.info("Datatype casting passed")
    return dataframe


def quality_check(dataframe):
    errors = []

    # 1. Required columns must not be null
    for column in ["trade_date", "stock_code"]:
        if dataframe[column].isna().any():
            errors.append(f"NULL found in required column '{column}': {dataframe[column].isna().sum()} rows")

    # 2. Required string values must not be empty
    empty_stock_code = dataframe["stock_code"].fillna("").str.strip().eq("")
    if empty_stock_code.any():
        errors.append(f"Empty stock_code: {empty_stock_code.sum()} rows")

    # 3. Duplicate grain
    grain_columns = ["trade_date", "stock_code"]
    duplicate_mask = dataframe.duplicated(subset=grain_columns, keep=False)
    if duplicate_mask.any():
        errors.append(f"Duplicate grain detected for ({', '.join(grain_columns)}): {duplicate_mask.sum()} rows")

    # 4. Non-negative values - IDX uses sentinels like -9999999, treat as NULL
    non_negative_columns = [
        "previous_price", "open_price", "first_trade_price", "high_price", "low_price", "close_price",
        "volume", "trading_value", "frequency",
        "offer_price", "offer_volume", "bid_price", "bid_volume",
        "listed_shares", "tradeable_shares",
        "foreign_sell", "foreign_buy",
        "non_regular_volume", "non_regular_value", "non_regular_frequency",
    ]
    for column in non_negative_columns:
        invalid_mask = dataframe[column] < 0
        if invalid_mask.any():
            logger.warning(f"Negative sentinel values in '{column}': {invalid_mask.sum()} rows. Converting to NULL.")
            dataframe.loc[invalid_mask, column] = pd.NA

    # 5. OHLC consistency (log only, not hard failures)
    ohlc_validations = {
        "High < Low": dataframe["high_price"] < dataframe["low_price"],
        "Open < Low": dataframe["open_price"] < dataframe["low_price"],
        "Open > High": dataframe["open_price"] > dataframe["high_price"],
        "Close < Low": dataframe["close_price"] < dataframe["low_price"],
        "Close > High": dataframe["close_price"] > dataframe["high_price"],
    }
    for rule, invalid_mask in ohlc_validations.items():
        if invalid_mask.sum() > 0:
            logger.warning(f"OHLC anomaly - {rule}: {invalid_mask.sum()} rows")

    # 6. Future trade date
    today = date.today()
    future_date_mask = dataframe["trade_date"] > today
    if future_date_mask.any():
        errors.append(f"trade_date in the future: {future_date_mask.sum()} rows")

    if errors:
        raise ValueError("Quality check failed:\n- " + "\n- ".join(errors))

    logger.info("Quality check passed")
    return dataframe


def transform_to_curated(dfs:dict):
    """Complete RAW -> CURATED transformation."""
    dfs_curated = {}
    logger.info("Starting RAW -> CURATED transformation")
    for key,value in dfs.items():
        dataframe = validate_source_schema(value)
        dataframe = rename_columns(dataframe)
        dataframe = dataframe[EXPECTED_CURATED_COLUMNS].copy()  # drops source-only fields (No, persen, etc.)
        dataframe = cast_datatypes(dataframe)
        dataframe = quality_check(dataframe)

        logger.info(f"Key : {key} \n RAW -> CURATED completed. Rows: {len(dataframe):,}, Columns: {len(dataframe.columns)}")
        dfs_curated[key] = dataframe
    return dfs_curated


# if __name__ == "__main__":
#     df_raw = load_raw(
#         datetime.strptime("2026-08-30", "%Y-%m-%d").date(),
#         datetime.strptime("2026-09-01", "%Y-%m-%d").date(),
#     )
#     df_curated = transform_to_curated(df_raw)
#     save_curated_to_s3(df_curated)

In [25]:
df_raw = load_raw(
    datetime.strptime("2021-01-01", "%Y-%m-%d").date(),
    datetime.strptime("2026-01-01", "%Y-%m-%d").date(),
)
# df_raw
df_curated = transform_to_curated(df_raw)
save_curated_to_s3(df_curated)

2026-09-15 11:46:14,563 - INFO - Loading raw: raw/idx/year=2021/month=01/data.parquet
2026-09-15 11:46:16,877 - INFO - Loading raw: raw/idx/year=2021/month=02/data.parquet
2026-09-15 11:46:18,685 - INFO - Loading raw: raw/idx/year=2021/month=03/data.parquet
2026-09-15 11:46:20,058 - INFO - Loading raw: raw/idx/year=2021/month=04/data.parquet
2026-09-15 11:46:21,444 - INFO - Loading raw: raw/idx/year=2021/month=05/data.parquet
2026-09-15 11:46:22,568 - INFO - Loading raw: raw/idx/year=2021/month=06/data.parquet
2026-09-15 11:46:24,105 - INFO - Loading raw: raw/idx/year=2021/month=07/data.parquet
2026-09-15 11:46:25,776 - INFO - Loading raw: raw/idx/year=2021/month=08/data.parquet
2026-09-15 11:46:27,746 - INFO - Loading raw: raw/idx/year=2021/month=09/data.parquet
2026-09-15 11:46:30,374 - INFO - Loading raw: raw/idx/year=2021/month=10/data.parquet
2026-09-15 11:46:33,934 - INFO - Loading raw: raw/idx/year=2021/month=11/data.parquet
2026-09-15 11:46:37,106 - INFO - Loading raw: raw/idx/

AttributeError: 'NoneType' object has no attribute 'columns'

In [32]:
df_raw = load_raw(
    datetime.strptime("2025-01-01", "%Y-%m-%d").date(),
    datetime.strptime("2026-01-01", "%Y-%m-%d").date(),
)
# df_raw
df_curated = transform_to_curated(df_raw)
save_curated_to_s3(df_curated)

2026-09-15 14:20:07,503 - INFO - Loading raw: raw/idx/year=2025/month=01/data.parquet
2026-09-15 14:20:09,538 - INFO - Loading raw: raw/idx/year=2025/month=02/data.parquet
2026-09-15 14:20:10,618 - INFO - Loading raw: raw/idx/year=2025/month=03/data.parquet
2026-09-15 14:20:11,606 - INFO - Loading raw: raw/idx/year=2025/month=04/data.parquet
2026-09-15 14:20:12,950 - INFO - Loading raw: raw/idx/year=2025/month=05/data.parquet
2026-09-15 14:20:14,116 - INFO - Loading raw: raw/idx/year=2025/month=06/data.parquet
2026-09-15 14:20:15,233 - INFO - Loading raw: raw/idx/year=2025/month=07/data.parquet
2026-09-15 14:20:16,584 - INFO - Loading raw: raw/idx/year=2025/month=08/data.parquet
2026-09-15 14:20:18,568 - INFO - Loading raw: raw/idx/year=2025/month=09/data.parquet
2026-09-15 14:20:20,118 - INFO - Loading raw: raw/idx/year=2025/month=10/data.parquet
2026-09-15 14:20:21,622 - INFO - Loading raw: raw/idx/year=2025/month=11/data.parquet
2026-09-15 14:20:22,985 - INFO - Loading raw: raw/idx/

In [92]:
s3 = boto3.client("s3")
response = s3.list_objects_v2(
    Bucket = BUCKET_NAME,
    Prefix = 'curated/idx'
)

for obj in response.get('Contents'):
    print(obj["Key"])
    

curated/idx/year=2020/month=01/data.parquet
curated/idx/year=2020/month=02/data.parquet
curated/idx/year=2020/month=03/data.parquet
curated/idx/year=2020/month=04/data.parquet
curated/idx/year=2020/month=05/data.parquet
curated/idx/year=2020/month=06/data.parquet
curated/idx/year=2020/month=07/data.parquet
curated/idx/year=2020/month=08/data.parquet
curated/idx/year=2020/month=09/data.parquet
curated/idx/year=2020/month=10/data.parquet
curated/idx/year=2020/month=11/data.parquet
curated/idx/year=2020/month=12/data.parquet
curated/idx/year=2021/month=01/data.parquet
curated/idx/year=2021/month=02/data.parquet
curated/idx/year=2021/month=03/data.parquet
curated/idx/year=2021/month=04/data.parquet
curated/idx/year=2021/month=05/data.parquet
curated/idx/year=2021/month=06/data.parquet
curated/idx/year=2021/month=07/data.parquet
curated/idx/year=2021/month=08/data.parquet
curated/idx/year=2021/month=09/data.parquet
curated/idx/year=2021/month=10/data.parquet
curated/idx/year=2021/month=11/d

In [94]:
response = s3.get_object(
    Bucket = BUCKET_NAME,
    Key = "curated/idx/year=2026/month=08/data.parquet"
)

df = pd.read_parquet(BytesIO(response["Body"].read()))
df.columns

Index(['stock_summary_id', 'trade_date', 'stock_code', 'stock_name', 'remarks',
       'delisting_date', 'previous_price', 'open_price', 'first_trade_price',
       'high_price', 'low_price', 'close_price', 'price_change', 'volume',
       'trading_value', 'frequency', 'individual_index', 'offer_price',
       'offer_volume', 'bid_price', 'bid_volume', 'listed_shares',
       'tradeable_shares', 'index_weight', 'foreign_sell', 'foreign_buy',
       'non_regular_volume', 'non_regular_value', 'non_regular_frequency'],
      dtype='object')

In [95]:
df["individual_index"]

0        164.0
1        572.7
2         66.3
3        799.1
4         67.7
         ...  
18292     68.0
18293     35.3
18294    106.9
18295    258.4
18296     52.0
Name: individual_index, Length: 18297, dtype: float64

In [103]:
response = s3.list_objects_v2(
    Bucket = BUCKET_NAME
)
response.get("Contents")

[{'Key': 'archived/idx/',
  'LastModified': datetime.datetime(2026, 9, 6, 16, 7, 59, tzinfo=tzutc()),
  'ETag': '"d41d8cd98f00b204e9800998ecf8427e"',
  'ChecksumAlgorithm': ['CRC64NVME'],
  'ChecksumType': 'FULL_OBJECT',
  'Size': 0,
  'StorageClass': 'STANDARD'},
 {'Key': 'curated/idx/year=2026/month=08/data.parquet',
  'LastModified': datetime.datetime(2026, 9, 13, 16, 17, 45, tzinfo=tzutc()),
  'ETag': '"c3f7a9808c73c8521a78622c63c9fd9e"',
  'ChecksumAlgorithm': ['CRC32'],
  'ChecksumType': 'FULL_OBJECT',
  'Size': 1221613,
  'StorageClass': 'STANDARD'},
 {'Key': 'curated/idx/year=2026/month=09/daily/date=2026-09-01/data.parquet',
  'LastModified': datetime.datetime(2026, 9, 13, 16, 17, 48, tzinfo=tzutc()),
  'ETag': '"6d634b85b3a2844aa78be712e3cba86e"',
  'ChecksumAlgorithm': ['CRC32'],
  'ChecksumType': 'FULL_OBJECT',
  'Size': 144648,
  'StorageClass': 'STANDARD'},
 {'Key': 'metadata/etl_control.json',
  'LastModified': datetime.datetime(2026, 9, 10, 16, 21, 26, tzinfo=tzutc()),


In [8]:
import duckdb
import logging
import pandas as pd
import boto3

from io import BytesIO
from datetime import datetime, timedelta, date
from botocore.exceptions import ClientError

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

BUCKET_NAME = "afdal-idx-stock-data-s3"
DIM_STOCK_KEY = "modeled/dim_stock/data.parquet"


def upload_df_to_s3(df, bucket_name, key, s3_client=None):
    s3_client = s3_client or boto3.client("s3")
    buffer = BytesIO()
    df.to_parquet(buffer, index=False, engine="pyarrow")
    s3_client.put_object(Bucket=bucket_name, Key=key, Body=buffer.getvalue())

def init_dim_sector_industry():
    #ini untuk sementara masih di-update manual mappingannya (mengikuti kebijakan IDX) dan menggunakan SDC type 1.
    key = "modeled/dim_sector_industry/data.parquet"
    idx_ic_flat_list = [
    {
        "code": "A111",
        "sector": "Energy",
        "sub_sector": "Oil, Gas & Coal",
        "industry": "Oil & Gas",
        "sub_industry": "Oil & Gas Production & Refinery"
    },
    {
        "code": "A112",
        "sector": "Energy",
        "sub_sector": "Oil, Gas & Coal",
        "industry": "Oil & Gas",
        "sub_industry": "Oil & Gas Storage & Distribution"
    },
    {
        "code": "A121",
        "sector": "Energy",
        "sub_sector": "Oil, Gas & Coal",
        "industry": "Coal",
        "sub_industry": "Coal Production"
    },
    {
        "code": "A122",
        "sector": "Energy",
        "sub_sector": "Oil, Gas & Coal",
        "industry": "Coal",
        "sub_industry": "Coal Distribution"
    },
    {
        "code": "A131",
        "sector": "Energy",
        "sub_sector": "Oil, Gas & Coal",
        "industry": "Oil, Gas & Coal Supports",
        "sub_industry": "Oil & Gas Drilling Service"
    },
    {
        "code": "A132",
        "sector": "Energy",
        "sub_sector": "Oil, Gas & Coal",
        "industry": "Oil, Gas & Coal Supports",
        "sub_industry": "Oil, Gas & Coal Equipment & Services"
    },
    {
        "code": "A211",
        "sector": "Energy",
        "sub_sector": "Alternative Energy",
        "industry": "Alternative Energy Equipment",
        "sub_industry": "Alternative Energy Equipment"
    },
    {
        "code": "A221",
        "sector": "Energy",
        "sub_sector": "Alternative Energy",
        "industry": "Alternative Fuels",
        "sub_industry": "Alternative Fuels"
    },
    {
        "code": "B111",
        "sector": "Basic Materials",
        "sub_sector": "Basic Materials",
        "industry": "Chemicals",
        "sub_industry": "Basic Chemicals"
    },
    {
        "code": "B112",
        "sector": "Basic Materials",
        "sub_sector": "Basic Materials",
        "industry": "Chemicals",
        "sub_industry": "Agricultural Chemicals"
    },
    {
        "code": "B113",
        "sector": "Basic Materials",
        "sub_sector": "Basic Materials",
        "industry": "Chemicals",
        "sub_industry": "Specialty Chemicals"
    },
    {
        "code": "B121",
        "sector": "Basic Materials",
        "sub_sector": "Basic Materials",
        "industry": "Construction Materials",
        "sub_industry": "Construction Materials"
    },
    {
        "code": "B131",
        "sector": "Basic Materials",
        "sub_sector": "Basic Materials",
        "industry": "Containers & Packaging",
        "sub_industry": "Containers & Packaging"
    },
    {
        "code": "B141",
        "sector": "Basic Materials",
        "sub_sector": "Basic Materials",
        "industry": "Metals & Minerals",
        "sub_industry": "Aluminum"
    },
    {
        "code": "B142",
        "sector": "Basic Materials",
        "sub_sector": "Basic Materials",
        "industry": "Metals & Minerals",
        "sub_industry": "Cooper"
    },
    {
        "code": "B143",
        "sector": "Basic Materials",
        "sub_sector": "Basic Materials",
        "industry": "Metals & Minerals",
        "sub_industry": "Gold"
    },
    {
        "code": "B144",
        "sector": "Basic Materials",
        "sub_sector": "Basic Materials",
        "industry": "Metals & Minerals",
        "sub_industry": "Iron & Steel"
    },
    {
        "code": "B145",
        "sector": "Basic Materials",
        "sub_sector": "Basic Materials",
        "industry": "Metals & Minerals",
        "sub_industry": "Precious Metals & Minerals"
    },
    {
        "code": "B146",
        "sector": "Basic Materials",
        "sub_sector": "Basic Materials",
        "industry": "Metals & Minerals",
        "sub_industry": "Diversified Metals & Minerals"
    },
    {
        "code": "B147",
        "sector": "Basic Materials",
        "sub_sector": "Basic Materials",
        "industry": "Metals & Minerals",
        "sub_industry": "Mining Equipment & Services"
    },
    {
        "code": "B151",
        "sector": "Basic Materials",
        "sub_sector": "Basic Materials",
        "industry": "Forestry & Paper",
        "sub_industry": "Timber"
    },
    {
        "code": "B152",
        "sector": "Basic Materials",
        "sub_sector": "Basic Materials",
        "industry": "Forestry & Paper",
        "sub_industry": "Paper"
    },
    {
        "code": "B153",
        "sector": "Basic Materials",
        "sub_sector": "Basic Materials",
        "industry": "Forestry & Paper",
        "sub_industry": "Diversified Forest Products"
    },
    {
        "code": "C111",
        "sector": "industryals",
        "sub_sector": "industryal Goods",
        "industry": "Aerospace & Defense",
        "sub_industry": "Aerospace & Defense"
    },
    {
        "code": "C121",
        "sector": "industryals",
        "sub_sector": "industryal Goods",
        "industry": "Building Products & Fixtures",
        "sub_industry": "Building Products & Fixtures"
    },
    {
        "code": "C131",
        "sector": "industryals",
        "sub_sector": "industryal Goods",
        "industry": "Electrical",
        "sub_industry": "Electrical Components & Equipment"
    },
    {
        "code": "C132",
        "sector": "industryals",
        "sub_sector": "industryal Goods",
        "industry": "Electrical",
        "sub_industry": "Heavy Electrical Equipment"
    },
    {
        "code": "C141",
        "sector": "industryals",
        "sub_sector": "industryal Goods",
        "industry": "Machinery",
        "sub_industry": "Construction Machinery & Heavy Vehicles"
    },
    {
        "code": "C142",
        "sector": "industryals",
        "sub_sector": "industryal Goods",
        "industry": "Machinery",
        "sub_industry": "Agricultural & Farm Machinery"
    },
    {
        "code": "C143",
        "sector": "industryals",
        "sub_sector": "industryal Goods",
        "industry": "Machinery",
        "sub_industry": "industryal Machinery & Components"
    },
    {
        "code": "C211",
        "sector": "industryals",
        "sub_sector": "industryal Services",
        "industry": "Diversified industryal Trading",
        "sub_industry": "Diversified industryal Trading"
    },
    {
        "code": "C221",
        "sector": "industryals",
        "sub_sector": "industryal Services",
        "industry": "Commercial Services",
        "sub_industry": "Commercial Printing"
    },
    {
        "code": "C222",
        "sector": "industryals",
        "sub_sector": "industryal Services",
        "industry": "Commercial Services",
        "sub_industry": "Environmental & Facilities Services"
    },
    {
        "code": "C223",
        "sector": "industryals",
        "sub_sector": "industryal Services",
        "industry": "Commercial Services",
        "sub_industry": "Office Supplies"
    },
    {
        "code": "C224",
        "sector": "industryals",
        "sub_sector": "industryal Services",
        "industry": "Commercial Services",
        "sub_industry": "Business Support Services"
    },
    {
        "code": "C231",
        "sector": "industryals",
        "sub_sector": "industryal Services",
        "industry": "Professional Services",
        "sub_industry": "Human Resource & Employment Services"
    },
    {
        "code": "C232",
        "sector": "industryals",
        "sub_sector": "industryal Services",
        "industry": "Professional Services",
        "sub_industry": "Research & Consulting Services"
    },
    {
        "code": "C311",
        "sector": "industryals",
        "sub_sector": "Multi-sector Holdings",
        "industry": "Multi-sector Holdings",
        "sub_industry": "Multi-sector Holdings"
    },
    {
        "code": "D111",
        "sector": "Consumer Non-Cyclicals",
        "sub_sector": "Food & Staples Retailing",
        "industry": "Food & Staples Retailing",
        "sub_industry": "Drug Retail & Distributors"
    },
    {
        "code": "D112",
        "sector": "Consumer Non-Cyclicals",
        "sub_sector": "Food & Staples Retailing",
        "industry": "Food & Staples Retailing",
        "sub_industry": "Food Retail & Distributors"
    },
    {
        "code": "D113",
        "sector": "Consumer Non-Cyclicals",
        "sub_sector": "Food & Staples Retailing",
        "industry": "Food & Staples Retailing",
        "sub_industry": "Supermarkets & Convenience Store"
    },
    {
        "code": "D211",
        "sector": "Consumer Non-Cyclicals",
        "sub_sector": "Food & Beverage",
        "industry": "Beverages",
        "sub_industry": "Liquors"
    },
    {
        "code": "D212",
        "sector": "Consumer Non-Cyclicals",
        "sub_sector": "Food & Beverage",
        "industry": "Beverages",
        "sub_industry": "Soft Drinks"
    },
    {
        "code": "D221",
        "sector": "Consumer Non-Cyclicals",
        "sub_sector": "Food & Beverage",
        "industry": "Processed Foods",
        "sub_industry": "Dairy Products"
    },
    {
        "code": "D222",
        "sector": "Consumer Non-Cyclicals",
        "sub_sector": "Food & Beverage",
        "industry": "Processed Foods",
        "sub_industry": "Processed Foods"
    },
    {
        "code": "D231",
        "sector": "Consumer Non-Cyclicals",
        "sub_sector": "Food & Beverage",
        "industry": "Agricultural Products",
        "sub_industry": "Fish, Meat, & Poultry"
    },
    {
        "code": "D232",
        "sector": "Consumer Non-Cyclicals",
        "sub_sector": "Food & Beverage",
        "industry": "Agricultural Products",
        "sub_industry": "Plantations & Crops"
    },
    {
        "code": "D311",
        "sector": "Consumer Non-Cyclicals",
        "sub_sector": "Tobacco",
        "industry": "Tobacco",
        "sub_industry": "Tobacco"
    },
    {
        "code": "D411",
        "sector": "Consumer Non-Cyclicals",
        "sub_sector": "Nondurable Household Products",
        "industry": "Household Products",
        "sub_industry": "Household Products"
    },
    {
        "code": "D421",
        "sector": "Consumer Non-Cyclicals",
        "sub_sector": "Nondurable Household Products",
        "industry": "Personal Care Products",
        "sub_industry": "Personal Care Products"
    },
    {
        "code": "E111",
        "sector": "Consumer Cyclicals",
        "sub_sector": "Automobiles & Components",
        "industry": "Auto Components",
        "sub_industry": "Auto Parts & Equipment"
    },
    {
        "code": "E112",
        "sector": "Consumer Cyclicals",
        "sub_sector": "Automobiles & Components",
        "industry": "Auto Components",
        "sub_industry": "Tires"
    },
    {
        "code": "E121",
        "sector": "Consumer Cyclicals",
        "sub_sector": "Automobiles & Components",
        "industry": "Automobiles",
        "sub_industry": "Car Manufacturers"
    },
    {
        "code": "E122",
        "sector": "Consumer Cyclicals",
        "sub_sector": "Automobiles & Components",
        "industry": "Automobiles",
        "sub_industry": "Motorcycle Manufacturers"
    },
    {
        "code": "E211",
        "sector": "Consumer Cyclicals",
        "sub_sector": "Household Goods",
        "industry": "Household Goods",
        "sub_industry": "Home Furnishings"
    },
    {
        "code": "E212",
        "sector": "Consumer Cyclicals",
        "sub_sector": "Household Goods",
        "industry": "Household Goods",
        "sub_industry": "Household Appliances"
    },
    {
        "code": "E213",
        "sector": "Consumer Cyclicals",
        "sub_sector": "Household Goods",
        "industry": "Household Goods",
        "sub_industry": "Housewares & Specialties"
    },
    {
        "code": "E311",
        "sector": "Consumer Cyclicals",
        "sub_sector": "Leisure Goods",
        "industry": "Consumer Electronics",
        "sub_industry": "Consumer Electronics"
    },
    {
        "code": "E321",
        "sector": "Consumer Cyclicals",
        "sub_sector": "Leisure Goods",
        "industry": "Sport Equipment & Hobbies Goods",
        "sub_industry": "Sport Equipment & Hobbies Goods"
    },
    {
        "code": "E411",
        "sector": "Consumer Cyclicals",
        "sub_sector": "Apparel & Luxury Goods",
        "industry": "Apparel & Luxury Goods",
        "sub_industry": "Clothing, Accessories & Bags"
    },
    {
        "code": "E412",
        "sector": "Consumer Cyclicals",
        "sub_sector": "Apparel & Luxury Goods",
        "industry": "Apparel & Luxury Goods",
        "sub_industry": "Footwear"
    },
    {
        "code": "E413",
        "sector": "Consumer Cyclicals",
        "sub_sector": "Apparel & Luxury Goods",
        "industry": "Apparel & Luxury Goods",
        "sub_industry": "Textiles"
    },
    {
        "code": "E511",
        "sector": "Consumer Cyclicals",
        "sub_sector": "Consumer Services",
        "industry": "Tourism & Recreation",
        "sub_industry": "Gaming Venue"
    },
    {
        "code": "E512",
        "sector": "Consumer Cyclicals",
        "sub_sector": "Consumer Services",
        "industry": "Tourism & Recreation",
        "sub_industry": "Hotels, Resorts & Cruise Lines"
    },
    {
        "code": "E513",
        "sector": "Consumer Cyclicals",
        "sub_sector": "Consumer Services",
        "industry": "Tourism & Recreation",
        "sub_industry": "Travel Agencies"
    },
    {
        "code": "E514",
        "sector": "Consumer Cyclicals",
        "sub_sector": "Consumer Services",
        "industry": "Tourism & Recreation",
        "sub_industry": "Recreational & Sports Facilities"
    },
    {
        "code": "E515",
        "sector": "Consumer Cyclicals",
        "sub_sector": "Consumer Services",
        "industry": "Tourism & Recreation",
        "sub_industry": "Restaurants"
    },
    {
        "code": "E521",
        "sector": "Consumer Cyclicals",
        "sub_sector": "Consumer Services",
        "industry": "Education & Support Services",
        "sub_industry": "Education Services"
    },
    {
        "code": "E522",
        "sector": "Consumer Cyclicals",
        "sub_sector": "Consumer Services",
        "industry": "Education & Support Services",
        "sub_industry": "Consumer Support Services"
    },
    {
        "code": "E611",
        "sector": "Consumer Cyclicals",
        "sub_sector": "Media & Entertainment",
        "industry": "Media",
        "sub_industry": "Advertising"
    },
    {
        "code": "E612",
        "sector": "Consumer Cyclicals",
        "sub_sector": "Media & Entertainment",
        "industry": "Media",
        "sub_industry": "Broadcasting"
    },
    {
        "code": "E613",
        "sector": "Consumer Cyclicals",
        "sub_sector": "Media & Entertainment",
        "industry": "Media",
        "sub_industry": "Cable & Satellite"
    },
    {
        "code": "E614",
        "sector": "Consumer Cyclicals",
        "sub_sector": "Media & Entertainment",
        "industry": "Media",
        "sub_industry": "Consumer Publishing"
    },
    {
        "code": "E621",
        "sector": "Consumer Cyclicals",
        "sub_sector": "Media & Entertainment",
        "industry": "Entertainment & Movie Production",
        "sub_industry": "Entertainment & Movie Production"
    },
    {
        "code": "E711",
        "sector": "Consumer Cyclicals",
        "sub_sector": "Retailing",
        "industry": "Consumer Distributors",
        "sub_industry": "Consumer Distributors"
    },
    {
        "code": "E721",
        "sector": "Consumer Cyclicals",
        "sub_sector": "Retailing",
        "industry": "Internet & Homeshop Retail",
        "sub_industry": "Internet & Homeshop Retail"
    },
    {
        "code": "E731",
        "sector": "Consumer Cyclicals",
        "sub_sector": "Retailing",
        "industry": "Department Stores",
        "sub_industry": "Department Stores"
    },
    {
        "code": "E741",
        "sector": "Consumer Cyclicals",
        "sub_sector": "Retailing",
        "industry": "Specialty Retail",
        "sub_industry": "Apparel & Textile Retail"
    },
    {
        "code": "E742",
        "sector": "Consumer Cyclicals",
        "sub_sector": "Retailing",
        "industry": "Specialty Retail",
        "sub_industry": "Electronics Retail"
    },
    {
        "code": "E743",
        "sector": "Consumer Cyclicals",
        "sub_sector": "Retailing",
        "industry": "Specialty Retail",
        "sub_industry": "Home Improvement Retail"
    },
    {
        "code": "E744",
        "sector": "Consumer Cyclicals",
        "sub_sector": "Retailing",
        "industry": "Specialty Retail",
        "sub_industry": "Specialty Stores"
    },
    {
        "code": "E745",
        "sector": "Consumer Cyclicals",
        "sub_sector": "Retailing",
        "industry": "Specialty Retail",
        "sub_industry": "Automotive Retail"
    },
    {
        "code": "F111",
        "sector": "Healthcare",
        "sub_sector": "Healthcare Equipment & Providers",
        "industry": "Healthcare Equipment & Supplies",
        "sub_industry": "Healthcare Equipment"
    },
    {
        "code": "F112",
        "sector": "Healthcare",
        "sub_sector": "Healthcare Equipment & Providers",
        "industry": "Healthcare Equipment & Supplies",
        "sub_industry": "Healthcare Supplies & Distributions"
    },
    {
        "code": "F121",
        "sector": "Healthcare",
        "sub_sector": "Healthcare Equipment & Providers",
        "industry": "Healthcare Providers",
        "sub_industry": "Healthcare Providers"
    },
    {
        "code": "F211",
        "sector": "Healthcare",
        "sub_sector": "Pharmaceuticals & Health Care Research",
        "industry": "Pharmaceuticals",
        "sub_industry": "Pharmaceuticals"
    },
    {
        "code": "F221",
        "sector": "Healthcare",
        "sub_sector": "Pharmaceuticals & Health Care Research",
        "industry": "Healthcare Research",
        "sub_industry": "Healthcare Research"
    },
    {
        "code": "G111",
        "sector": "Financials",
        "sub_sector": "Banks",
        "industry": "Banks",
        "sub_industry": "Banks"
    },
    {
        "code": "G211",
        "sector": "Financials",
        "sub_sector": "Financing Service",
        "industry": "Consumer Financing",
        "sub_industry": "Consumer Financing"
    },
    {
        "code": "G221",
        "sector": "Financials",
        "sub_sector": "Financing Service",
        "industry": "Business Financing",
        "sub_industry": "Venture Capital"
    },
    {
        "code": "G222",
        "sector": "Financials",
        "sub_sector": "Financing Service",
        "industry": "Business Financing",
        "sub_industry": "Specialize Business Financing"
    },
    {
        "code": "G311",
        "sector": "Financials",
        "sub_sector": "Investment Service",
        "industry": "Investment Services",
        "sub_industry": "Investment Management"
    },
    {
        "code": "G312",
        "sector": "Financials",
        "sub_sector": "Investment Service",
        "industry": "Investment Services",
        "sub_industry": "Investment Banking & Brokerage Services"
    },
    {
        "code": "G313",
        "sector": "Financials",
        "sub_sector": "Investment Service",
        "industry": "Investment Services",
        "sub_industry": "Market Operators"
    },
    {
        "code": "G314",
        "sector": "Financials",
        "sub_sector": "Investment Service",
        "industry": "Investment Services",
        "sub_industry": "Investment Support Service"
    },
    {
        "code": "G411",
        "sector": "Financials",
        "sub_sector": "Insurance",
        "industry": "Insurance",
        "sub_industry": "Insurance Brokers"
    },
    {
        "code": "G412",
        "sector": "Financials",
        "sub_sector": "Insurance",
        "industry": "Insurance",
        "sub_industry": "General Insurance"
    },
    {
        "code": "G413",
        "sector": "Financials",
        "sub_sector": "Insurance",
        "industry": "Insurance",
        "sub_industry": "Life Insurance"
    },
    {
        "code": "G414",
        "sector": "Financials",
        "sub_sector": "Insurance",
        "industry": "Insurance",
        "sub_industry": "Reinsurance"
    },
    {
        "code": "G511",
        "sector": "Financials",
        "sub_sector": "Holding & Investment Companies",
        "industry": "Holding & Investment Companies",
        "sub_industry": "Financial Holdings"
    },
    {
        "code": "G512",
        "sector": "Financials",
        "sub_sector": "Holding & Investment Companies",
        "industry": "Holding & Investment Companies",
        "sub_industry": "Investment Companies"
    },
    {
        "code": "H111",
        "sector": "Properties & Real Estate",
        "sub_sector": "Properties & Real Estate",
        "industry": "Real Estate Management & Development",
        "sub_industry": "Real Estate Development & Management"
    },
    {
        "code": "H112",
        "sector": "Properties & Real Estate",
        "sub_sector": "Properties & Real Estate",
        "industry": "Real Estate Management & Development",
        "sub_industry": "Real Estate Services"
    },
    {
        "code": "I111",
        "sector": "Technology",
        "sub_sector": "Software & IT Services",
        "industry": "Online Applications & Services",
        "sub_industry": "Online Applications & Services"
    },
    {
        "code": "I121",
        "sector": "Technology",
        "sub_sector": "Software & IT Services",
        "industry": "IT Services & Consulting",
        "sub_industry": "IT Services & Consulting"
    },
    {
        "code": "I131",
        "sector": "Technology",
        "sub_sector": "Software & IT Services",
        "industry": "Software",
        "sub_industry": "Software"
    },
    {
        "code": "I211",
        "sector": "Technology",
        "sub_sector": "Technology Hardware & Equipment",
        "industry": "Networking Equipment",
        "sub_industry": "Networking Equipment"
    },
    {
        "code": "I221",
        "sector": "Technology",
        "sub_sector": "Technology Hardware & Equipment",
        "industry": "Computer Hardware",
        "sub_industry": "Computer Hardware"
    },
    {
        "code": "I231",
        "sector": "Technology",
        "sub_sector": "Technology Hardware & Equipment",
        "industry": "Electronic Equipment, Instruments & Components",
        "sub_industry": "Electronic Equipment & Instruments"
    },
    {
        "code": "I232",
        "sector": "Technology",
        "sub_sector": "Technology Hardware & Equipment",
        "industry": "Electronic Equipment, Instruments & Components",
        "sub_industry": "Electronic Components & Semiconductors"
    },
    {
        "code": "J111",
        "sector": "Infrastructures",
        "sub_sector": "Transportation Infrastructure",
        "industry": "Transport Infrastructure Operator",
        "sub_industry": "Airport Operators"
    },
    {
        "code": "J112",
        "sector": "Infrastructures",
        "sub_sector": "Transportation Infrastructure",
        "industry": "Transport Infrastructure Operator",
        "sub_industry": "Highways & Railtracks"
    },
    {
        "code": "J113",
        "sector": "Infrastructures",
        "sub_sector": "Transportation Infrastructure",
        "industry": "Transport Infrastructure Operator",
        "sub_industry": "Marine Ports & Services"
    },
    {
        "code": "J211",
        "sector": "Infrastructures",
        "sub_sector": "Heavy Constructions & Civil Engineering",
        "industry": "Heavy Constructions & Civil Engineering",
        "sub_industry": "Heavy Constructions & Civil Engineering"
    },
    {
        "code": "J311",
        "sector": "Infrastructures",
        "sub_sector": "Telecommunication",
        "industry": "Telecommunication Service",
        "sub_industry": "Wired Telecommunication Service"
    },
    {
        "code": "J312",
        "sector": "Infrastructures",
        "sub_sector": "Telecommunication",
        "industry": "Telecommunication Service",
        "sub_industry": "Integrated Telecommunication Service"
    },
    {
        "code": "J321",
        "sector": "Infrastructures",
        "sub_sector": "Telecommunication",
        "industry": "Wireless Telecommunication Services",
        "sub_industry": "Wireless Telecommunication Services"
    },
    {
        "code": "J411",
        "sector": "Infrastructures",
        "sub_sector": "Utilities",
        "industry": "Electric Utilities",
        "sub_industry": "Electric Utilities"
    },
    {
        "code": "J421",
        "sector": "Infrastructures",
        "sub_sector": "Utilities",
        "industry": "Gas Utilities",
        "sub_industry": "Gas Utilities"
    },
    {
        "code": "J431",
        "sector": "Infrastructures",
        "sub_sector": "Utilities",
        "industry": "Water Utilities",
        "sub_industry": "Water Utilities"
    },
    {
        "code": "K111",
        "sector": "Transportation & Logistic",
        "sub_sector": "Transportation",
        "industry": "Airlines",
        "sub_industry": "Airlines"
    },
    {
        "code": "K121",
        "sector": "Transportation & Logistic",
        "sub_sector": "Transportation",
        "industry": "Passenger Marine Transportation",
        "sub_industry": "Passenger Marine Transportation"
    },
    {
        "code": "K131",
        "sector": "Transportation & Logistic",
        "sub_sector": "Transportation",
        "industry": "Passenger Land Transportation",
        "sub_industry": "Rail"
    },
    {
        "code": "K132",
        "sector": "Transportation & Logistic",
        "sub_sector": "Transportation",
        "industry": "Passenger Land Transportation",
        "sub_industry": "Road Transportation"
    },
    {
        "code": "K211",
        "sector": "Transportation & Logistic",
        "sub_sector": "Logistics & Deliveries",
        "industry": "Logistics & Deliveries",
        "sub_industry": "Logistics & Deliveries"
    },
    {
        "code": "Z111",
        "sector": "Listed Investment Product",
        "sub_sector": "Investment Trusts",
        "industry": "Investment Trusts",
        "sub_industry": "Mutual Fund/ETFS"
    },
    {
        "code": "Z112",
        "sector": "Listed Investment Product",
        "sub_sector": "Investment Trusts",
        "industry": "Investment Trusts",
        "sub_industry": "Real Estate Investment Trusts"
    },
    {
        "code": "Z113",
        "sector": "Listed Investment Product",
        "sub_sector": "Investment Trusts",
        "industry": "Investment Trusts",
        "sub_industry": "Infrastructure Investment Trusts"
    },
    {
        "code": "Z211",
        "sector": "Listed Investment Product",
        "sub_sector": "Bonds",
        "industry": "Bonds",
        "sub_industry": "Government Bonds"
    },
    {
        "code": "Z212",
        "sector": "Listed Investment Product",
        "sub_sector": "Bonds",
        "industry": "Bonds",
        "sub_industry": "Corporate Bonds"
    }
]
    df_sector_industry = pd.DataFrame(idx_ic_flat_list)
    upload_df_to_s3(df_sector_industry,BUCKET_NAME,key)

def init_dim_date():
    # Rentangnya bisa disesuaikan, dan cukup sekali diinitial load aja.
    start_date = "2020-01-01"
    end_date = "2040-12-31"
    date_range = pd.date_range(start=start_date, end=end_date)

    dim_date = pd.DataFrame({"Date": date_range})

    dim_date["date_key"] = dim_date["Date"].dt.strftime("%Y%m%d").astype(int)
    dim_date["date"] = dim_date["Date"].dt.date
    dim_date["year"] = dim_date["Date"].dt.year
    dim_date["month"] = dim_date["Date"].dt.month
    dim_date["quarter"] = dim_date["Date"].dt.quarter
    dim_date["day_name"] = dim_date["Date"].dt.day_name()

    dim_date = dim_date.drop(columns=["Date"])

    key = "modeled/dim_date/data.parquet"
    upload_df_to_s3(dim_date, BUCKET_NAME, key)

def _get_duckdb_connection():
    con = duckdb.connect()
    con.execute("INSTALL httpfs;")
    con.execute("LOAD httpfs;")
    con.execute("""
        CREATE OR REPLACE SECRET (
            TYPE S3,
            PROVIDER credential_chain
        );
    """)
    return con

def init_dim_stock():
    # One-time init: bangun dim_stock SCD2 dari seluruh histori curated (2020 s.d. bulan terakhir yang sudah tercompact)
    con = _get_duckdb_connection()
    source = f"s3://{BUCKET_NAME}/curated/idx/year=*/month=*/data.parquet"

    dim_stock = con.execute(
        f"""
        WITH changes AS (
            SELECT
                trade_date,
                stock_code,
                stock_name,
                COALESCE(
                    stock_name != LAG(stock_name) OVER (
                        PARTITION BY stock_code ORDER BY trade_date
                    ),
                    TRUE
                ) AS name_changed
            FROM read_parquet('{source}', hive_partitioning = true)
--            WHERE year >= 2020
--              AND (year < 2026 OR (year = 2026 AND CAST(month AS INTEGER) <= 7))
        ), last_date AS (
            SELECT MAX(trade_date) as last_update_date
            FROM changes
        ),
        versions AS (
            SELECT *,
                SUM(CAST(name_changed AS INTEGER)) OVER (
                    PARTITION BY stock_code ORDER BY trade_date
                    ROWS UNBOUNDED PRECEDING
                ) AS version
            FROM changes
        ),
        history AS (
            SELECT
                stock_code,
                version,
                FIRST(stock_name ORDER BY trade_date) AS stock_name,
                MIN(trade_date) AS valid_from,
                MAX(trade_date) AS valid_to
            FROM versions
            GROUP BY stock_code, version
        )
        SELECT
            stock_code,
            stock_name,
            valid_from,
            CASE
                WHEN ROW_NUMBER() OVER (PARTITION BY stock_code ORDER BY valid_from DESC) = 1
                THEN NULL ELSE valid_to
            END AS valid_to,
            (ROW_NUMBER() OVER (PARTITION BY stock_code ORDER BY valid_from DESC) = 1)::INTEGER AS is_active,
            (SELECT *
            FROM last_date
            ) as last_update_date
        FROM history
        ORDER BY stock_code, valid_from
        """
    ).df()

    logger.info(
        f"init_dim_stock: generated {len(dim_stock)} rows for "
        f"{dim_stock['stock_code'].nunique()} stocks"
    )
    upload_df_to_s3(dim_stock, BUCKET_NAME, DIM_STOCK_KEY)

def update_dim_stock(key):
    # SCD2 incremental update dim_stock
    con = _get_duckdb_connection()
    dim_stock_path = f"s3://{BUCKET_NAME}/{DIM_STOCK_KEY}"
    new_data_path = ", ".join(f"'s3://{BUCKET_NAME}/{path}'" for path in key)
    # new_data_path = (
    #     f"s3://{BUCKET_NAME}/{key}"
    # )

    df_stock = con.execute(
        f"""
        WITH dim_current AS (
            SELECT * FROM read_parquet('{dim_stock_path}')
        ),
        current_active AS (
            SELECT stock_code, stock_name AS current_name, valid_from AS current_valid_from, last_update_date
            FROM dim_current
            WHERE is_active = 1
        ),
        get_last_update AS (
            SELECT MAX(last_update_date) as last_update_date
            FROM current_active
        ),
        new_changes AS (
            SELECT
                trade_date,
                stock_code,
                stock_name,
                COALESCE(
                    stock_name != LAG(stock_name) OVER (
                        PARTITION BY stock_code ORDER BY trade_date
                    ),
                    TRUE
                ) AS name_changed
            FROM read_parquet([{new_data_path}], union_by_name = True)
            WHERE trade_date>(SELECT * FROM get_last_update)
        ),
        new_versions AS (
            SELECT *,
                SUM(CAST(name_changed AS INTEGER)) OVER (
                    PARTITION BY stock_code ORDER BY trade_date
                    ROWS UNBOUNDED PRECEDING
                ) AS version
            FROM new_changes
        ),
        new_segments AS (
            SELECT
                stock_code,
                version,
                FIRST(stock_name ORDER BY trade_date) AS stock_name,
                MIN(trade_date) AS seg_start,
                MAX(trade_date) AS seg_end,
                MAX(version) OVER (PARTITION BY stock_code) AS max_version
            FROM new_versions
            GROUP BY stock_code, version
        ),
        merged AS (
            -- segmen pertama tiap stock di batch baru, digabung sama record aktif eksisting
            SELECT
                s.stock_code,
                s.stock_name,
                CASE WHEN s.stock_name = c.current_name THEN c.current_valid_from ELSE s.seg_start END AS valid_from,
                CASE WHEN s.version = s.max_version THEN NULL ELSE s.seg_end END AS valid_to
            FROM new_segments s
            LEFT JOIN current_active c ON s.stock_code = c.stock_code
            WHERE s.version = 1

            UNION ALL

            -- ganti nama yang kejadian di tengah2 batch baru
            SELECT
                s.stock_code,
                s.stock_name,
                s.seg_start AS valid_from,
                CASE WHEN s.version = s.max_version THEN NULL ELSE s.seg_end END AS valid_to
            FROM new_segments s
            WHERE s.version > 1

            UNION ALL

            -- tutup record aktif lama kalau nama udah beda dari H1 batch baru
            SELECT
                c.stock_code,
                c.current_name AS stock_name,
                c.current_valid_from AS valid_from,
                s.seg_start - 1 AS valid_to
            FROM new_segments s
            JOIN current_active c ON s.stock_code = c.stock_code
            WHERE s.version = 1 AND s.stock_name != c.current_name

            UNION ALL

            -- stock tanpa data baru di batch ini (suspend dll): biarin apa adanya
            SELECT
                c.stock_code,
                c.current_name AS stock_name,
                c.current_valid_from AS valid_from,
                NULL AS valid_to
            FROM current_active c
            WHERE NOT EXISTS (
                SELECT 1 FROM new_segments s WHERE s.stock_code = c.stock_code
            )
        ),
        final_active AS (
            SELECT
                stock_code,
                stock_name,
                valid_from,
                valid_to,
                (ROW_NUMBER() OVER (PARTITION BY stock_code ORDER BY valid_from DESC) = 1)::INTEGER AS is_active
            FROM merged
        )
        SELECT stock_code, stock_name, valid_from, valid_to, is_active FROM final_active
        UNION ALL
        SELECT stock_code, stock_name, valid_from, valid_to, is_active FROM dim_current WHERE is_active = 0
        ORDER BY stock_code, valid_from
        """
    ).df()
    # cek ada perubahan beneran atau engga, sebelum upload
    changed = con.execute(
        """
        SELECT COUNT(*) FROM (
            (SELECT * FROM df_stock EXCEPT 
                SELECT stock_code, stock_name, valid_from, valid_to, is_active 
                FROM read_parquet(?))
            UNION ALL
            (SELECT stock_code, stock_name, valid_from, valid_to, is_active 
            FROM read_parquet(?) 
            EXCEPT SELECT * FROM df_stock)
        )
        """,
        [dim_stock_path, dim_stock_path],
    ).fetchone()[0]

    if changed == 0:
        logger.info(f"update_dim_stock: no changes, skip upload")
        return 

    df_last_update = con.execute(
            f"""
                SELECT max(trade_date) as last_update FROM read_parquet([{new_data_path}], union_by_name = True)
            """).fetchone()[0]

    df_stock["last_update_date"] = df_last_update
    logger.info(f"update_dim_stock: {changed} row(s), uploading")
    # return df_stock
    upload_df_to_s3(df_stock, BUCKET_NAME, DIM_STOCK_KEY)


In [ ]:
import logging
from io import BytesIO
from datetime import datetime, timedelta, date

import boto3
import pandas as pd
from botocore.exceptions import ClientError

BUCKET_NAME = "afdal-idx-stock-data-s3"

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

# Mapping column
STANDARDIZE_COLUMN_MAP = {
    "IDStockSummary": "stock_summary_id",
    "Date": "trade_date",
    "StockCode": "stock_code",
    "StockName": "stock_name",
    "Remarks": "remarks",
    "DelistingDate": "delisting_date",

    "Previous": "previous_price",
    "OpenPrice": "open_price",
    "FirstTrade": "first_trade_price",
    "High": "high_price",
    "Low": "low_price",
    "Close": "close_price",
    "Change": "price_change",

    "Volume": "volume",
    "Value": "trading_value",
    "Frequency": "frequency",

    "IndexIndividual": "individual_index",

    "Offer": "offer_price",
    "OfferVolume": "offer_volume",
    "Bid": "bid_price",
    "BidVolume": "bid_volume",

    "ListedShares": "listed_shares",
    "TradebleShares": "tradeable_shares",
    "WeightForIndex": "index_weight",

    "ForeignSell": "foreign_sell",
    "ForeignBuy": "foreign_buy",

    "NonRegularVolume": "non_regular_volume",
    "NonRegularValue": "non_regular_value",
    "NonRegularFrequency": "non_regular_frequency",
}

EXPECTED_SOURCE_COLUMNS = set(STANDARDIZE_COLUMN_MAP.keys())
EXPECTED_CURATED_COLUMNS = list(STANDARDIZE_COLUMN_MAP.values())

# Data type
INTEGER_COLUMNS = [
    "stock_summary_id", "volume", "frequency", "offer_volume", "bid_volume",
    "listed_shares", "tradeable_shares", "foreign_sell", "foreign_buy",
    "non_regular_volume", "non_regular_frequency",
]

FLOAT_COLUMNS = [
    "previous_price", "open_price", "first_trade_price", "high_price", "low_price",
    "close_price", "price_change", "trading_value", "individual_index",
    "offer_price", "bid_price", "index_weight", "non_regular_value",
]

STRING_COLUMNS = ["stock_code", "stock_name", "remarks"]
DATE_COLUMNS = ["trade_date", "delisting_date"]


# S3
def read_df_from_s3(bucket_name, key, s3_client=None):
    s3_client = s3_client or boto3.client("s3")
    try:
        response = s3_client.get_object(
            Bucket=bucket_name,
            Key=key,
        )
    except ClientError as e:
        if e.response["Error"]["Code"] in ("NoSuchKey", "404"):
            return None
        raise

    return pd.read_parquet(BytesIO(response["Body"].read()))



def upload_df_to_s3(df, bucket_name, key, s3_client=None):
    s3_client = s3_client or boto3.client("s3")
    buffer = BytesIO()
    df.to_parquet(buffer, index=False, engine="pyarrow")
    s3_client.put_object(Bucket=bucket_name, Key=key, Body=buffer.getvalue())


def save_curated_to_s3(dfs:dict):
    s3_client = boto3.client("s3")
    for key,value in dfs.items():
        curated_key = key.replace("raw/", "curated/", 1)
        logger.info(f"Uploading curated: {curated_key} ({len(value)} rows)")
        upload_df_to_s3(value, BUCKET_NAME, curated_key, s3_client)


def get_month_list(start_date, end_date):
    months = set()
    current_date = start_date
    while current_date <= end_date:
        months.add(current_date.year * 100 + current_date.month)
        current_date += timedelta(days=1)
    return sorted(months)


def load_raw(start_date, end_date):
    """Load raw data for a date range and return it as one concatenated DataFrame."""
    months = get_month_list(start_date, end_date)
    s3_client = boto3.client("s3")
    dfs = {}

    for i, year_month in enumerate(months):
        month, year = year_month % 100, year_month // 100
        mm, yyyy = f"{month:02d}", f"{year:04d}"

        if i != len(months) - 1:
            key = f"raw/idx/year={yyyy}/month={mm}/data.parquet"
            logger.info(f"Loading raw: {key}")
            dfs[key] = read_df_from_s3(BUCKET_NAME, key, s3_client)
        else:
            first_date = date(year, month, 1) if len(months) > 1 else start_date
            for current_date in pd.date_range(first_date, end_date):
                current_date = current_date.date()
                key = f"raw/idx/year={yyyy}/month={mm}/daily/date={current_date}/data.parquet"
                logger.info(f"Loading raw: {key}")

                df = read_df_from_s3(BUCKET_NAME, key, s3_client)
                if df is not None:
                    dfs[key] = df
                else:
                    logger.info(f"No data found: {key}")

    logger.info(f"Loaded {len(dfs)} raw files")
    return dfs


# Validation
def validate_source_schema(dataframe):
    actual_columns = set(dataframe.columns)
    missing_columns = EXPECTED_SOURCE_COLUMNS - actual_columns
    unexpected_columns = actual_columns - EXPECTED_SOURCE_COLUMNS

    if missing_columns:
        raise ValueError(f"Source schema validation failed. Missing columns: {sorted(missing_columns)}")

    if unexpected_columns:
        logger.warning(f"Unexpected source columns detected: {sorted(unexpected_columns)}")

    logger.info("Source schema validation passed")
    return dataframe


# Rename
def rename_columns(dataframe):
    return dataframe.rename(columns=STANDARDIZE_COLUMN_MAP)


# Data type casting
def cast_numeric_column(dataframe, column, dtype):
    original = dataframe[column]
    original_null = original.isna()
    converted = pd.to_numeric(original, errors="coerce")

    conversion_error_mask = ~original_null & converted.isna()
    conversion_error_count = conversion_error_mask.sum()

    if conversion_error_count > 0:
        bad_values = original[conversion_error_mask].unique()
        raise ValueError(
            f"Datatype conversion failed for column '{column}'. "
            f"{conversion_error_count} invalid values. Examples: {bad_values[:5]}"
        )

    dataframe[column] = converted.astype(dtype)
    return dataframe


def cast_datatypes(dataframe):
    """Apply the curated data type contract."""
    for column in INTEGER_COLUMNS:
        dataframe = cast_numeric_column(dataframe, column, "Int64")

    for column in FLOAT_COLUMNS:
        dataframe = cast_numeric_column(dataframe, column, "float64")

    for column in STRING_COLUMNS:
        dataframe[column] = dataframe[column].astype("string").str.strip()

    for column in DATE_COLUMNS:
        dataframe[column] = pd.to_datetime(dataframe[column], errors="coerce").dt.date

    logger.info("Datatype casting passed")
    return dataframe


def quality_check(dataframe):
    errors = []

    # 1. Required columns must not be null
    for column in ["trade_date", "stock_code"]:
        if dataframe[column].isna().any():
            errors.append(f"NULL found in required column '{column}': {dataframe[column].isna().sum()} rows")

    # 2. Required string values must not be empty
    empty_stock_code = dataframe["stock_code"].fillna("").str.strip().eq("")
    if empty_stock_code.any():
        errors.append(f"Empty stock_code: {empty_stock_code.sum()} rows")

    # 3. Duplicate grain
    grain_columns = ["trade_date", "stock_code"]
    duplicate_mask = dataframe.duplicated(subset=grain_columns, keep=False)
    if duplicate_mask.any():
        errors.append(f"Duplicate grain detected for ({', '.join(grain_columns)}): {duplicate_mask.sum()} rows")

    # 4. Non-negative values - IDX uses sentinels like -9999999, treat as NULL
    non_negative_columns = [
        "previous_price", "open_price", "first_trade_price", "high_price", "low_price", "close_price",
        "volume", "trading_value", "frequency",
        "offer_price", "offer_volume", "bid_price", "bid_volume",
        "listed_shares", "tradeable_shares",
        "foreign_sell", "foreign_buy",
        "non_regular_volume", "non_regular_value", "non_regular_frequency",
    ]
    for column in non_negative_columns:
        invalid_mask = dataframe[column] < 0
        if invalid_mask.any():
            logger.warning(f"Negative sentinel values in '{column}': {invalid_mask.sum()} rows. Converting to NULL.")
            dataframe.loc[invalid_mask, column] = pd.NA

    # 5. OHLC consistency (log only, not hard failures)
    ohlc_validations = {
        "High < Low": dataframe["high_price"] < dataframe["low_price"],
        "Open < Low": dataframe["open_price"] < dataframe["low_price"],
        "Open > High": dataframe["open_price"] > dataframe["high_price"],
        "Close < Low": dataframe["close_price"] < dataframe["low_price"],
        "Close > High": dataframe["close_price"] > dataframe["high_price"],
    }
    for rule, invalid_mask in ohlc_validations.items():
        if invalid_mask.sum() > 0:
            logger.warning(f"OHLC anomaly - {rule}: {invalid_mask.sum()} rows")

    # 6. Future trade date
    today = date.today()
    future_date_mask = dataframe["trade_date"] > today
    if future_date_mask.any():
        errors.append(f"trade_date in the future: {future_date_mask.sum()} rows")

    if errors:
        raise ValueError("Quality check failed:\n- " + "\n- ".join(errors))

    logger.info("Quality check passed")
    return dataframe


def transform_to_curated(dfs:dict):
    """Complete RAW -> CURATED transformation."""
    dfs_curated = {}
    logger.info("Starting RAW -> CURATED transformation")
    for key,value in dfs.items():
        dataframe = validate_source_schema(value)
        dataframe = rename_columns(dataframe)
        dataframe = dataframe[EXPECTED_CURATED_COLUMNS].copy()  # drops source-only fields (No, persen, etc.)
        dataframe = cast_datatypes(dataframe)
        dataframe = quality_check(dataframe)

        logger.info(f"Key : {key} \n RAW -> CURATED completed. Rows: {len(dataframe):,}, Columns: {len(dataframe.columns)}")
        dfs_curated[key] = dataframe
    return dfs_curated


# if __name__ == "__main__":
#     df_raw = load_raw(
#         datetime.strptime("2026-08-30", "%Y-%m-%d").date(),
#         datetime.strptime("2026-09-01", "%Y-%m-%d").date(),
#     )
#     df_curated = transform_to_curated(df_raw)
#     save_curated_to_s3(df_curated)

In [100]:
def get_month_list(start_date, end_date):
    months = set()
    current_date = start_date
    while current_date <= end_date:
        months.add(current_date.year * 100 + current_date.month)
        current_date += timedelta(days=1)
    return sorted(months)


# S3
def read_df_from_s3(bucket_name, key, s3_client=None):
    s3_client = s3_client or boto3.client("s3")
    try:
        response = s3_client.get_object(
            Bucket=bucket_name,
            Key=key,
        )
    except ClientError as e:
        if e.response["Error"]["Code"] in ("NoSuchKey", "404"):
            return None
        raise

    return pd.read_parquet(BytesIO(response["Body"].read()))


def upload_df_to_s3(df, bucket_name, key, s3_client=None):
    s3_client = s3_client or boto3.client("s3")
    buffer = BytesIO()
    df.to_parquet(buffer, index=False, engine="pyarrow")
    s3_client.put_object(Bucket=bucket_name, Key=key, Body=buffer.getvalue())


def load_curated(start_date, end_date):
    """Load curated data for a date range and return it as one concatenated DataFrame."""
    months = get_month_list(start_date, end_date)
    s3_client = boto3.client("s3")
    dfs = {}

    for i, year_month in enumerate(months):
        month, year = year_month % 100, year_month // 100
        mm, yyyy = f"{month:02d}", f"{year:04d}"

        if i != len(months) - 1:
            key = f"curated/idx/year={yyyy}/month={mm}/data.parquet"
            logger.info(f"Loading curated: {key}")
            dfs[key] = read_df_from_s3(BUCKET_NAME, key, s3_client)
        else:
            first_date = date(year, month, 1)
            for current_date in pd.date_range(first_date, end_date):
                current_date = current_date.date()
                key = f"curated/idx/year={yyyy}/month={mm}/daily/date={current_date}/data.parquet"
                logger.info(f"Loading curated: {key}")

                df = read_df_from_s3(BUCKET_NAME, key, s3_client)
                if df is not None:
                    dfs[key] = df
                else:
                    logger.info(f"No data found: {key}")

    logger.info(f"Loaded {len(dfs)} curated files")
    return dfs


def _get_duckdb_connection():
    con = duckdb.connect()
    con.execute("INSTALL httpfs;")
    con.execute("LOAD httpfs;")
    con.execute("""
        CREATE OR REPLACE SECRET (
            TYPE S3,
            PROVIDER credential_chain
        );
    """)
    return con


def fact_stock_daily(yyyy=None, mm=None, source_df=None):
    con = _get_duckdb_connection()

    if source_df is None:
        source = f"""
            read_parquet(
                's3://{BUCKET_NAME}/curated/idx/year={yyyy}/month={mm}/data.parquet'
            )
        """
    else:
        con.register("source_df", source_df)
        source = "source_df"

    return con.execute(
        f"""
        SELECT
            CAST(strftime(trade_date, '%Y%m%d') AS INTEGER) AS date_key,
            stock_code,
            regexp_extract(
                remarks,'00([A-Z][0-9]{{3}})',1
            ) AS idx_ic_code,
            previous_price,
            open_price,
            first_trade_price,
            high_price,
            low_price,
            close_price,
            price_change,
            volume,
            trading_value,
            frequency,
            foreign_buy,
            foreign_sell,
            non_regular_volume,
            non_regular_value,
            non_regular_frequency
        FROM {source}
        """
    ).df()


In [106]:
import re 

start_date = datetime.strptime("2025-01-01", "%Y-%m-%d").date()
end_date = datetime.strptime("2026-01-01", "%Y-%m-%d").date()
dfs = load_curated(start_date,end_date)
months = get_month_list(start_date,end_date)
for key,value in dfs.items():
    # months = get_month_list(start_date, end_date)
    df_daily = []
    if "daily" in key:
        df_daily.append(value)
        continue

    yyyy = re.search(r"year=(\d{4})", key).group(1)
    mm = re.search(r"month=(\d{2})", key).group(1)
    fact_key = f"modeled/fact_stock_daily/year={yyyy}/month={mm}/data.parquet"
    df = fact_stock_daily(yyyy,mm)
    upload_df_to_s3(df, BUCKET_NAME, fact_key)
    logger.info(f"Uploading: {fact_key} (DONE)")

if df_daily:
    df_daily = pd.concat(df_daily, ignore_index=True)    
    month, year = months[-1] % 100, months[-1] // 100
    mm, yyyy = f"{month:02d}", f"{year:04d}"    
    fact_key = f"modeled/fact_stock_daily/year={yyyy}/month={mm}/data.parquet"
    df = fact_stock_daily(yyyy,mm, df_daily)
    upload_df_to_s3(df, BUCKET_NAME, fact_key)
    logger.info(f"Uploading: {fact_key} (DONE)")

2026-09-18 15:54:12,624 - INFO - Loading curated: curated/idx/year=2025/month=01/data.parquet


2026-09-18 15:54:16,863 - INFO - Loading curated: curated/idx/year=2025/month=02/data.parquet
2026-09-18 15:54:19,316 - INFO - Loading curated: curated/idx/year=2025/month=03/data.parquet
2026-09-18 15:54:22,174 - INFO - Loading curated: curated/idx/year=2025/month=04/data.parquet
2026-09-18 15:54:24,103 - INFO - Loading curated: curated/idx/year=2025/month=05/data.parquet
2026-09-18 15:54:27,298 - INFO - Loading curated: curated/idx/year=2025/month=06/data.parquet
2026-09-18 15:54:29,893 - INFO - Loading curated: curated/idx/year=2025/month=07/data.parquet
2026-09-18 15:54:33,135 - INFO - Loading curated: curated/idx/year=2025/month=08/data.parquet
2026-09-18 15:54:36,278 - INFO - Loading curated: curated/idx/year=2025/month=09/data.parquet
2026-09-18 15:54:39,085 - INFO - Loading curated: curated/idx/year=2025/month=10/data.parquet
2026-09-18 15:54:42,452 - INFO - Loading curated: curated/idx/year=2025/month=11/data.parquet
2026-09-18 15:54:45,562 - INFO - Loading curated: curated/id

In [20]:
def _get_duckdb_connection():
    con = duckdb.connect()
    con.execute("INSTALL httpfs;")
    con.execute("LOAD httpfs;")
    con.execute("""
        CREATE OR REPLACE SECRET (
            TYPE S3,
            PROVIDER credential_chain
        );
    """)
    return con
con = _get_duckdb_connection()

In [73]:
df_idx = con.execute(
    f""" 
    select 
        cast(strftime(d1.trade_date, '%Y%m%d') as integer) as date_key,
        d1.stock_code,
        regexp_extract(d1.remarks, '00([A-Z][0-9]{{3}})', 1) as idx_ic_code,
        open_price,
        high_price,
        low_price,
        close_price,
        volume,
        trading_value,
        frequency,
        foreign_buy,
        foreign_sell
    from read_parquet(
    's3://{BUCKET_NAME}/curated/idx/year=2026/month=08/data.parquet' 
    ) as d1
    """
).df()

In [74]:
df_idx

,date_key,stock_code,idx_ic_code,open_price,high_price,low_price,close_price,volume,trading_value,frequency,foreign_buy,foreign_sell
0,20260803,AADI,A121,9300.0,9300.0,9075.0,9100.0,4529500,4.143086e+10,3273,503600,2396100
1,20260803,AALI,D232,6900.0,7175.0,6825.0,7050.0,2232900,1.572849e+10,2078,738100,1111300
2,20260803,ABBA,E614,0.0,0.0,0.0,30.0,0,0.000000e+00,0,0,0
3,20260803,ABDA,G412,0.0,3530.0,3490.0,3500.0,10100,3.542400e+07,10,0,0
4,20260803,ABMM,A121,2550.0,2580.0,2520.0,2540.0,667500,1.699334e+09,409,197600,14600
...,...,...,...,...,...,...,...,...,...,...,...,...
18292,20260831,ZATA,E741,67.0,69.0,65.0,68.0,49491700,3.332512e+09,2902,8877000,7435000
18293,20260831,ZBRA,C311,0.0,0.0,0.0,50.0,0,0.000000e+00,0,0,0
18294,20260831,ZINC,B146,0.0,0.0,0.0,30.0,0,0.000000e+00,0,0,0
18295,20260831,ZONE,E741,0.0,780.0,700.0,770.0,82200,6.229050e+07,85,6300,0


In [66]:
df_idx[df_idx["suspension_status"] == 1]["stock_code"].unique()

array(['ABBA', 'ACST', 'ADES', 'AEGS', 'AGAR', 'AKKU', 'ALMI', 'ALTO',
       'AMMS', 'ANDI', 'ARKA', 'ARMY', 'ARTI', 'ASMI', 'BAPI', 'BATA',
       'BAUT', 'BEBS', 'BEKS', 'BHIT', 'BIKA', 'BIMA', 'BLTA', 'BLTZ',
       'BMBL', 'BOSS', 'BSWD', 'BTEK', 'BTEL', 'CANI', 'CASH', 'CBMF',
       'CHIP', 'CMPP', 'CNKO', 'COAL', 'COWL', 'CPRI', 'DEAL', 'DIGI',
       'DUCK', 'DUTI', 'ELTY', 'ENVY', 'ETWA', 'EURO', 'FASW', 'FIMP',
       'FLMC', 'FOOD', 'GAMA', 'GLOB', 'GOLL', 'GRPM', 'HADE', 'HAJJ',
       'HBAT', 'HILL', 'HITS', 'HKMU', 'HOME', 'HOTL', 'HRME', 'IBFN',
       'IBOS', 'IBST', 'IDEA', 'IFSH', 'IIKP', 'IKAI', 'INAF', 'INCF',
       'INPS', 'INRU', 'INTA', 'IPAC', 'IPPE', 'IPTV', 'ISAP', 'JSKY',
       'KARW', 'KAYU', 'KBAG', 'KBRI', 'KIAS', 'KING', 'KKES', 'KLIN',
       'KOIN', 'KREN', 'LCGP', 'LFLO', 'LMAS', 'LMAX', 'LMSH', 'LOPI',
       'LUCY', 'MABA', 'MAGP', 'MANG', 'MAPB', 'MARI', 'MDRN', 'MEJA',
       'MENN', 'MFMI', 'MGLV', 'MIRA', 'MKNT', 'MLIA', 'MPPA', 'MSIE',
      

In [67]:
df_idx[df_idx["stock_code"]=='UDNG']

,date_key,stock_code,idx_ic_code,suspension_status,open_price,high_price,low_price,close_price,volume,trading_value,...,listed_shares,tradeable_shares,index_weight,foreign_sell_1,foreign_buy_1,non_regular_volume,non_regular_value,non_regular_frequency,month,year
908,20260803,UDNG,D231,1,0.0,1175.0,1060.0,1090.0,669900,7.402115e+08,...,1750040072,1750040072,0.0,90200,48600,0,0.0,0,08,2026
1871,20260804,UDNG,D231,1,0.0,1195.0,1070.0,1195.0,844200,9.649690e+08,...,1750040072,1750040072,0.0,69200,48800,0,0.0,0,08,2026
2834,20260805,UDNG,D231,1,0.0,1290.0,1160.0,1245.0,789900,9.782145e+08,...,1750040072,1750040072,0.0,116700,25900,0,0.0,0,08,2026
3797,20260806,UDNG,D231,1,0.0,1290.0,1195.0,1250.0,874900,1.093756e+09,...,1750040072,1750040072,0.0,41400,26100,0,0.0,0,08,2026
4760,20260807,UDNG,D231,1,0.0,1375.0,1200.0,1375.0,2426100,3.264208e+09,...,1750040072,1750040072,0.0,11300,726900,0,0.0,0,08,2026
5723,20260810,UDNG,D231,1,0.0,1510.0,1340.0,1420.0,1692800,2.414888e+09,...,1750040072,1750040072,0.0,181400,162900,1000,1420000.0,1,08,2026
6686,20260811,UDNG,D231,1,0.0,1430.0,1280.0,1290.0,484600,6.414895e+08,...,1750040072,1750040072,0.0,10200,22900,0,0.0,0,08,2026
7649,20260812,UDNG,D231,1,0.0,1330.0,1180.0,1220.0,1231000,1.524133e+09,...,1750040072,1750040072,0.0,128600,47500,0,0.0,0,08,2026
8612,20260813,UDNG,D231,1,0.0,1250.0,1180.0,1200.0,439700,5.249010e+08,...,1750040072,1750040072,0.0,64700,0,0,0.0,0,08,2026
9575,20260814,UDNG,D231,1,0.0,1300.0,1170.0,1300.0,1067100,1.330499e+09,...,1750040072,1750040072,0.0,58500,40300,0,0.0,0,08,2026


In [89]:
con.execute(
    f""" 
    select 
        *
    from read_parquet(
    's3://{BUCKET_NAME}/modeled/fact_stock_daily/year=2026/month=08/data.parquet' 
    ) as d1
    """
).df()

,date_key,stock_code,idx_ic_code,open_price,high_price,low_price,close_price,volume,trading_value,frequency,foreign_buy,foreign_sell,month,year
0,20260803,AADI,A121,9300.0,9300.0,9075.0,9100.0,4529500,4.143086e+10,3273,503600,2396100,08,2026
1,20260803,AALI,D232,6900.0,7175.0,6825.0,7050.0,2232900,1.572849e+10,2078,738100,1111300,08,2026
2,20260803,ABBA,E614,0.0,0.0,0.0,30.0,0,0.000000e+00,0,0,0,08,2026
3,20260803,ABDA,G412,0.0,3530.0,3490.0,3500.0,10100,3.542400e+07,10,0,0,08,2026
4,20260803,ABMM,A121,2550.0,2580.0,2520.0,2540.0,667500,1.699334e+09,409,197600,14600,08,2026
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18292,20260831,ZATA,E741,67.0,69.0,65.0,68.0,49491700,3.332512e+09,2902,8877000,7435000,08,2026
18293,20260831,ZBRA,C311,0.0,0.0,0.0,50.0,0,0.000000e+00,0,0,0,08,2026
18294,20260831,ZINC,B146,0.0,0.0,0.0,30.0,0,0.000000e+00,0,0,0,08,2026
18295,20260831,ZONE,E741,0.0,780.0,700.0,770.0,82200,6.229050e+07,85,6300,0,08,2026
